[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GSTT-CSC/summer-school-tutorials/blob/main/day2_qms_regs/QMS_requirements_and_design.ipynb)

# CSC Summer School Day 2 — Quality Management Systems & Regulations

## Clinical context: HandOsteo

**HandOsteo** is an as-yet unreleased Software as a Medical Device (SaMD) that automatically screens patients for osteoporosis and osteopenia from routine AP (anteroposterior) hand X-rays. It is currently in development by the GSTT-CSC Team

Osteoporosis is a skeletal disease characterised by reduced bone mineral density (BMD), affecting approximately 3.5 million people in the UK and causing around 500,000 fragility fractures per year.<sup>[[1]](#ref1)</sup> Early detection is critical — patients with known osteoporosis can be treated with bisphosphonates or lifestyle interventions that substantially reduce fracture risk.<sup>[[2]](#ref2)</sup>

A validated, low-cost screening proxy for BMD is the **Metacarpal Cortical Percentage (MCP)**:<sup>[[3]](#ref3)</sup> the ratio of cortical bone thickness to total bone width across the shaft of the second metacarpal, measured on a plain AP hand X-ray.

```
        |<------- A (total width) ------->|
        |<- B/2 ->|            |<- B/2 ->|
        ┌─────────┬────────────┬──────────┐
        │ cortex  │  medullary │  cortex  │
        └─────────┴────────────┴──────────┘
        
        MCP = ((A − B) / A) × 100
```

A MCP below 60 % is associated with osteopenia; below 50 % with osteoporosis.<sup>[[4,5]](#ref4)</sup>

### The HandOsteo pipeline

We have created a very basic copy of what the HandOsteo pipeline might look like:

| Step | Class | What it does |
|------|-------|--------------|
| 1 | `DicomLoader` | Load and validate the modality of the DICOM (must be CR or DX) |
| 2 | `ViewClassifier` | Confirm the view is AP or PA; reject oblique views |
| 3 | `MCPMeasurer` | Segment the second metacarpal and compute A, B, and MCP |
| 4 | `ReportGenerator` | Create a structured report and route it to PACS |

---

## What you will do today

You will act as both the developer and the quality engineer for HandOsteo and produce a lightweight **Quality Management System (QMS)**, based on the [GSTT CSC QMS template](https://github.com/GSTT-CSC/QMS-Template):

| # | Stage | What you will produce |
|---|-------|-----------------------|
| 1 | **Project Initiation** | Device description, classification, and role assignments |
| 2 | **Requirements Gathering** | System Requirements Specification (SRS) from the care process |
| 3 | **Hazard Log** | Risk assessment linked back to requirements |
| 4 | **Design Specification** | System Design Specification (SDS) items that implement each SRS |
| 5 | **Verification** | Automated unit tests that confirm the code meets the SDS |
| 6 | **Validation** | Manual test cases that confirm the system meets the SRS |

All artefacts are stored as YAML and rendered into QMS documents via a `make` pipeline — mirroring how the GSTT CSC QMS Template works.

---

### References

<a id="ref1"></a>1. Royal Osteoporosis Society. *Media Toolkit — Facts and Statistics*. Available at: https://theros.org.uk/about-us/media-centre/media-toolkit/

<a id="ref2"></a>2. National Institute for Health and Care Excellence. *Bisphosphonates for treating osteoporosis*. Technology appraisal TA464. London: NICE; 2018. Available at: https://www.nice.org.uk/guidance/ta464

<a id="ref3"></a>3. Barnett E, Nordin BEC. The radiological diagnosis of osteoporosis: a new approach. *Clin Radiol*. 1960;11(3):166–174. doi:10.1016/S0009-9260(60)80012-8

<a id="ref4"></a>4. Kim JW, Kim JK, Lim KH, et al. The 2nd metacarpal cortical index as a simple screening tool for osteopenia. *J Bone Metab*. 2020;27(4):261–266. doi:10.11005/jbm.2020.27.4.261 — [PMC7746479](https://pmc.ncbi.nlm.nih.gov/articles/PMC7746479/)

<a id="ref5"></a>5. Schreiber JJ, Anderson PA, Hsu WK. Simple assessment of global bone density and osteoporosis screening using standard radiographs of the hand. *J Hand Surg Am*. 2017;42(4):e272–e279. doi:10.1016/j.jhsa.2017.01.012 — [PMID 28242242](https://pubmed.ncbi.nlm.nih.gov/28242242/)

In [ ]:
# Make sure we have all our dependencies installed in Colab
%pip install -q rdm pandas pytest pyyaml pydicom ipytest pillow dicom2nifti nibabel svgwrite ipython

In [ ]:
import os
import sys

# Check if we have the data and helper files, and if not, clone them from the repo
if not os.path.isdir("day2_data") or not os.path.exists("src/notebook_helpers.py"):
    !git clone --depth=1 -b main https://github.com/GSTT-CSC/summer-school-tutorials.git _repo
    !cp -rn _repo/day2_qms_regs/day2_data .
    !cp -rn _repo/day2_qms_regs/src .
    !rm -rf _repo

# Add the current directory to the Python path if it's not already there
if "." not in sys.path:
    sys.path.insert(0, ".")

print("Data ready:", os.path.isdir("day2_data"))
print("Helpers ready:", os.path.exists("src/notebook_helpers.py"))

# The boring display boilerplate (YAML printing, the diagrams, document rendering)
# lives in src/notebook_helpers.py so the cells below stay focused on the QMS.
from src.notebook_helpers import (
    show_yaml,
    show_care_diagram,
    show_doc,
    show_qms_process_flow,
    show_handosteo_pipeline,
)

### HandOsteo Care Process Diagram

The diagram below shows a proposed care pathway for HandOsteo — automated osteoporosis/osteopenia screening from AP hand X-rays.

In [ ]:
show_care_diagram()

### The HandOsteo pipeline

The four classes from the table above, shown end to end: `DicomLoader` loads and validates
the DICOM, `ViewClassifier` confirms the view is AP/PA, `MCPMeasurer` measures the second
metacarpal, and `ReportGenerator` produces the screening report routed back to PACS.

In [ ]:
# The four HandOsteo pipeline classes, with the X-ray each stage works on.
show_handosteo_pipeline()

### The QMS traceability chain

Everything you produce today links into one chain. A **risk control** and a **requirement**
are implemented by a **system design spec**, which is then **verified** by unit tests and
**validated** by manual tests. The stages map onto the *"What you will do today"* table
above — keep this chain in mind as you fill in each artefact.

In [ ]:
# The QMS traceability chain you will build across both notebooks.
show_qms_process_flow()

> ### ✏️ How to edit files in Colab
>
> Several steps below ask you to edit a `.yml` file. In Colab:
>
> 1. Click the **📁 Files** icon in the left sidebar.
> 2. Open the **`day2_data/data/`** folder.
> 3. **Double-click** a file (e.g. `device.yml`) to open it in the editor pane on the right.
> 4. Make your edits, then press **Ctrl/Cmd-S** to save.
>
> ⚠️ Colab storage is temporary — if your runtime disconnects you'll lose your edits and the data, and need to re-run the setup cells above.

> ### 📐 About the worked examples
>
> Each data file already contains **one completed example** — but notice it describes a deliberately *unrelated* device: **EarthMeasurer**, Eratosthenes' 2,000-year-old method for estimating the circumference of the Earth from the angle of a shadow.
>
> That's on purpose. An unrelated example shows you the *shape* of a good requirement, hazard, design item, or test — without handing you the HandOsteo answers. Your job is to add the **HandOsteo** entries yourself, using the care process diagram above.
>
> Genuinely stuck? The full worked HandOsteo records are one toggle away 👇

> ### 💡 Stuck? Load the example solutions
>
> If you'd like to see a worked example for HandOsteo, run the cell below.
> It will overwrite the files in `day2_data/data/` with completed versions — you can then read through them
> and use them as a reference for the rest of the tutorial.
>
> ⚠️ This will replace anything you have already written in the data files.

In [ ]:
# 💡 Load the worked HandOsteo solutions into day2_data/data/
# Set LOAD_SOLUTIONS = True and run this cell only if you want to replace your work with the example answers.

LOAD_SOLUTIONS = False  # <-- change to True to load solutions

if LOAD_SOLUTIONS:
    import shutil
    from pathlib import Path

    for src in Path("day2_data/data_solutions").iterdir():
        shutil.copy(src, f"day2_data/data/{src.name}")
    print("Solutions loaded — refresh the file browser (↺) to see them.")
else:
    print(
        "LOAD_SOLUTIONS is False — nothing changed. Set it to True above if you want the example answers."
    )

# 1. Project Initiation

Open `day2_data/data/device.yml`. It currently holds the placeholder **EarthMeasurer** device — replace its details with **HandOsteo**'s:
- A **product name** and **version number**
- The **device classification** (class I, IIa, IIb, or III under UK MDR 2002)

Then open `day2_data/data/people.yml` and assign roles (Developer, Quality Engineer, Clinical Lead).

In [ ]:
# Display the contents of the device and people yaml files.
print("Device Data:")
show_yaml("day2_data/data/device.yml")
print("People Data:")
show_yaml("day2_data/data/people.yml")

# 2. Requirements Gathering

Open `day2_data/data/requirements.yml` and add a new system requirement for HandOsteo, based on the care process diagram above. An example requirement (for the unrelated EarthMeasurer device) is already in the file to show the structure.

Each requirement needs:
- A short **title** and **description**
- A **unique identifier** (e.g. `SRS_002`)
- A **requirement type**

### Rules for writing good requirements

- **Clear, concise, and unambiguous** — one interpretation only
- **Measurable** — must be verifiable by an automated test
- Focus on **what** the software must do, not how it does it
- Use the word **MUST** (not "shall", which is ambiguous)

In [ ]:
# Display the contents of the requirements yaml file.
show_yaml("day2_data/data/requirements.yml")

# 3. Hazard Log

Open `day2_data/data/risk.yml` and add a new hazard for HandOsteo based on the care process diagram.
An example hazard (again for the EarthMeasurer device) has been provided to show the structure.

### Rules

- **Hazard** — the potential source of harm (not the harm itself).
- **Causes** — what could lead to the hazardous situation.
- **Effects** — what happens as a result of the hazard.
- **Harm** — the impact on the patient or user.
- **Severity / Likelihood** — use the labels defined at the top of the file.
- **Risk control** — a mitigation that reduces the likelihood or severity to an acceptable level.
- Link your risk control back to a requirement using a `linked_risk_control` field in `requirements.yml`.

In [ ]:
# Display the contents of the risk yaml file.
show_yaml("day2_data/data/risk.yml")

# 4. Design Specification

A requirement says *what* the software must do; a design specification says *how* you intend to satisfy it — in enough detail to build and test against.

Open `day2_data/data/requirements.yml` and add a design specification item (`sys_des_spec`) for the requirement you wrote in step 2. Link it back with the `linked_reqs` field so the requirement → design → test chain stays traceable.

A good design spec item:
- Has **enough detail** to implement the feature correctly
- Does **not** prescribe specific function names or class structures (unless a particular framework is required, e.g. "use MONAI")
- Is written so that it can be **verified by an automated test**

In [ ]:
# Display the contents of the requirements yaml file.
show_yaml("day2_data/data/requirements.yml")

### Checkpoint: render the QMS documents

The YAML files in `day2_data/data/` are the source records. The `make` command turns them into the controlled markdown documents in `day2_data/release/`.

Run the build cell, then preview each generated document. Check that your requirement, hazard, and SDS item read coherently as a traceable set before you start writing tests.

> ### ✏️ Optional: add your own prose to the documents
>
> Each rendered document is built from **two** sources:
>
> 1. **Your YAML data** (`day2_data/data/`) — this fills the tables (requirements, hazards, design items, SOUP).
> 2. **A template** (`day2_data/docs/`) — this provides the surrounding narrative: purpose, scope, introduction, users, use cases, architecture, and so on.
>
> The templates ship with placeholder prose marked `[TODO: ...]` and `[EG: ...]`. You can leave them as-is and the documents still render — **or**, if you want a more complete record, open the template and replace the placeholders with HandOsteo-specific text:
>
> | Document | Template to edit | Placeholder sections you could fill in |
> |---|---|---|
> | System Requirements | `day2_data/docs/system-requirements.md` | Introduction, Users, Use Environments, Use Cases |
> | System Design Spec | `day2_data/docs/system-design-specification.md` | Architecture diagrams, Software Items, UI Mockups |
> | Hazard Log | `day2_data/docs/hazard-log.md` | (fully data-driven — nothing to add by hand) |
>
> **Two things to keep in mind when editing a template:**
> - Leave the `{{ ... }}` tags alone (e.g. `{{device.name}}`, `{%- for ... %}`) — these pull live values from your YAML when the document is rendered. Edit the plain text around them.
> - Re-run the **`make`** cell below after editing to see your changes flow into `day2_data/release/`.
>
> This step is entirely optional — skip it if you'd rather focus on the data records.

In [ ]:
# Build the documentation using the command 'make'
! cd day2_data && make && cd ..

In [ ]:
# 📄 See your completed QMS rendered into official documents
show_doc("day2_data/release/system-requirements.md")

In [ ]:
# 📄 See your completed QMS rendered into official documents
show_doc("day2_data/release/hazard-log.md")

In [ ]:
# 📄 See your completed QMS rendered into official documents
show_doc("day2_data/release/system-design-specification.md")

In [ ]:
# 📄 See your completed QMS rendered into official documents
show_doc("day2_data/release/medical-device-classification.md")

---
## ✅ Requirements & design complete — next: verification & validation

You've produced the front half of the QMS: device record, requirements, hazard log, and
design specification, all rendered into controlled documents.

**Continue in [`QMS_testing_and_validation.ipynb`](QMS_testing_and_validation.ipynb)** to:

- **Verify** the software against your design specs by writing unit tests, and
- **Validate** it against the clinical need with manual test cases.

That notebook loads a complete worked QMS automatically, so you can start testing straight away.